# CSV to SSMS Database Uploader

This notebook performs the following steps:
1. Connects to a SQL Server (SSMS) instance.
2. Reads a CSV file into a Pandas DataFrame.
3. Automatically maps Pandas data types to SQL Server compatible types.
4. Creates a table and uploads the data.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, types
import urllib
import os

# --- 1. CONFIGURATION ---
# Update these values with your SSMS details
SERVER = 'DESKTOP-AR905LJ'       # e.g., 'localhost\\SQLEXPRESS' or 'DESKTOP-XXXX'
DATABASE = 'staging'   # The target database
TABLE_NAME = 'RFM_unified_data'   # Name of the table to create
CSV_FILE_PATH = '../raw/RFM_unified_data.csv'  # Path to your CSV file

# --- 2. CONNECTION SETUP ---
# We use Windows Authentication (Trusted_Connection=yes)
# If using SQL Auth, use: "UID=user;PWD=password;"
connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID=sa;"
    f"PWD=ghom3220;"
)

params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

print("Connection engine created successfully.")

Connection engine created successfully.


In [2]:
# --- 3. LOAD DATA ---
if os.path.exists(CSV_FILE_PATH):
    df = pd.read_csv(CSV_FILE_PATH)
    print(f"Loaded {len(df)} rows from {CSV_FILE_PATH}")
    display(df.head())
else:
    print(f"Error: File {CSV_FILE_PATH} not found.")

Loaded 25000 rows from ../raw/RFM_unified_data.csv


,ClientID,NomClient,Sexe,Age,TrancheAge,Ville,Region,DateInscription,AncienneteJours,AncienneteAnnees,...,ScoreRFM,SegmentRFM,SegmentDetail,ActionRecommandee,Satisfaction,NPS,Commentaire,Campagne,CodePromo,PointsFidelite
0,CLT01848,Chokri Chaabane,M,44,35-45,Tozeur,Sud,2023-10-27,50,0.1,...,145,EnSommeil,Anciens clients actifs autrefois,Campagne de réactivation - Offre spéciale,Très insatisfait,8,NaN,Soldes_été,PROMO38,1291
1,CLT03844,Rami Gharbi,M,40,35-45,Bizerte,Nord,2022-11-30,863,2.4,...,311,ÀSurveiller,Comportement mixte,Observation et segmentation affinée,Très insatisfait,8,Livraison rapide,Soldes_été,NaN,190
2,CLT00902,Rim Masmoudi,F,69,65+,Siliana,Nord,2023-12-07,531,1.5,...,335,Fidèles,Clients fidèles avec bon potentiel,Upselling et cross-selling,Neutre,6,NaN,Nouvel_An,NaN,754
3,CLT00448,Jamel Dhraief,M,62,55-65,Ariana,Grand Tunis,2022-10-30,4,0.0,...,125,Perdus,Gros dépensiers inactifs,Win-back campaign - Offre aggressive,Insatisfait,0,NaN,Fête_des_mères,NaN,2100
4,CLT00674,Lotfi Lahmar,M,27,25-35,Kasserine,Centre,2020-02-15,1453,4.0,...,135,EnSommeil,Anciens clients actifs autrefois,Campagne de réactivation - Offre spéciale,Neutre,10,NaN,Fête_des_mères,NaN,991


In [ ]:
# --- 4. DATA TYPE MAPPING ---
def generate_sql_types(df):
    """
    Maps Pandas dtypes to SQLAlchemy types to ensure compatibility with SSMS.
    """
    d_type_map = {}
    for col, dtype in df.dtypes.items():
        if "int" in str(dtype):
            d_type_map[col] = types.BigInteger() if "64" in str(dtype) else types.Integer()
        elif "float" in str(dtype):
            d_type_map[col] = types.Float()
        elif "datetime" in str(dtype):
            d_type_map[col] = types.DateTime()
        elif "bool" in str(dtype):
            d_type_map[col] = types.Boolean()
        else:
            # Determine max length for strings to avoid NVARCHAR(MAX) if not needed
            max_len = df[col].astype(str).str.len().max()
            if pd.isna(max_len) or max_len == 0:
                max_len = 255
            
            if max_len > 4000:
                d_type_map[col] = types.NVARCHAR(length=None) # NVARCHAR(MAX)
            else:
                d_type_map[col] = types.NVARCHAR(length=int(max_len))
                
    return d_type_map

sql_column_types = generate_sql_types(df)
print("Generated SQL column type mapping.")

In [ ]:
# --- 5. UPLOAD TO DATABASE ---
try:
    df.to_sql(
        name=TABLE_NAME,
        con=engine,
        if_exists='replace', # 'replace' creates the table or overwrites it
        index=False,         # Don't upload the DataFrame index as a column
        dtype=sql_column_types
    )
    print(f"Successfully created table '{TABLE_NAME}' and uploaded data.")
except Exception as e:
    print(f"Failed to upload data: {e}")